# RPM별 응답 계산 — 교차 확인용 코랩 노트북

웹 도구(https://aebonlee.github.io/data09-04/)와 **같은 계산**을 파이썬으로 합니다. 결과를 웹 도구의 엑셀과 대조하는 용도입니다.

- 가진 주파수 f = 차수 × RPM / 60 (Hz) — 가정: 회전 차수
- 차수 응답 = |FRF(f)| × 가진력 (원문 참고 1)
- overall = √(Σ 차수 응답²) (원문 참고 2, RSS)

저장소의 `samples/예시데이터_FRF.csv` 는 **가상 예시 데이터**입니다.

## 1. FRF CSV 올리기
첫 행 머리행, 첫 열 주파수(Hz). 응답점 열은 `이름` / `이름 Mag`+`이름 Phase` / `이름 Real`+`이름 Imag` 중 하나로 찾습니다.

In [ ]:
from google.colab import files
up = files.upload()
FRF_CSV = next(iter(up))
FRF_CSV

In [ ]:
import argparse
import csv
import math
import sys


def read_frf(path, point):
    with open(path, encoding="utf-8-sig", newline="") as fp:
        rows = list(csv.reader(fp))
    head = [h.strip() for h in rows[0]]

    def col(name):
        return head.index(name) if name in head else -1

    if col(point) >= 0:
        fmt, a, b = "mag", col(point), -1
    elif col(point + " Mag") >= 0:
        fmt, a, b = "magphase", col(point + " Mag"), col(point + " Phase")
    elif col(point + " Real") >= 0:
        fmt, a, b = "reim", col(point + " Real"), col(point + " Imag")
    else:
        sys.exit("응답점 열을 찾지 못했습니다: " + point)
    recs = {}
    for r in rows[1:]:
        try:
            f = float(r[0])
            va = float(r[a])
            vb = float(r[b]) if b >= 0 else 0.0
        except (ValueError, IndexError):
            continue
        if f in recs:
            continue  # 같은 주파수는 처음 값만 (웹 도구와 같음)
        recs[f] = math.hypot(va, vb) if fmt == "reim" else abs(va)
    freq = sorted(recs)
    return freq, [recs[f] for f in freq]


def interpolate(freq, vals, f, method="linear"):
    if not (freq[0] <= f <= freq[-1]):
        return None
    lo, hi = 0, len(freq) - 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if freq[mid] <= f:
            lo = mid
        else:
            hi = mid
    if freq[lo] == f:
        return vals[lo]
    if freq[hi] == f:
        return vals[hi]
    f0, f1, v0, v1 = freq[lo], freq[hi], vals[lo], vals[hi]
    if method == "nearest":
        return v0 if f - f0 <= f1 - f else v1
    t = (f - f0) / (f1 - f0)
    if method == "loglog" and f0 > 0 and v0 > 0 and v1 > 0:
        tl = math.log(f / f0) / math.log(f1 / f0)
        return math.exp(math.log(v0) + tl * (math.log(v1) - math.log(v0)))
    return v0 + t * (v1 - v0)


def rpm_list(start, end, step):
    n = int(math.floor((end - start) / step + 1e-9)) + 1
    out = [round(start + i * step, 9) for i in range(n)]
    if round(out[-1], 6) < round(end, 6):
        out.append(end)
    return out


def scale_ratios(orders, scales, ref):
    """F_k = (s_k / s_ref) × F_ref 의 비 s_k / s_ref"""
    if len(scales) != len(orders):
        sys.exit("차수와 scale factor 개수가 다릅니다.")
    if ref not in orders:
        sys.exit("기준 차수가 차수 목록에 없습니다.")
    if any(s <= 0 for s in scales):
        sys.exit("scale factor 는 0 보다 커야 합니다.")
    sref = scales[orders.index(ref)]
    return {k: s / sref for k, s in zip(orders, scales)}


def read_force_vector(path, keys):
    """RPM 연동 가진력 벡터 CSV → (rpm 오름차순, {차수: [값…]}). 숫자가 아닌 머리행은 건너뜁니다."""
    with open(path, encoding="utf-8-sig", newline="") as fp:
        recs = []
        for r in csv.reader(fp):
            try:
                recs.append((float(r[0]), [float(r[j + 1]) for j in range(len(keys))]))
            except (ValueError, IndexError):
                continue
    recs.sort()
    return [r for r, _ in recs], {k: [v[j] for _, v in recs] for j, k in enumerate(keys)}


# ── 최소제곱 (Householder QR) — js/logic.js 의 lstsq 와 같은 방법 ─────────────
def lstsq(A, b):
    m, n = len(A), len(A[0])
    if m < n:
        return None
    R = [row[:] for row in A]
    y = b[:]
    for j in range(n):
        norm = math.sqrt(sum(R[i][j] ** 2 for i in range(j, m)))
        if norm == 0:
            continue
        alpha = -norm if R[j][j] > 0 else norm
        v = [R[i][j] for i in range(j, m)]
        v[0] -= alpha
        vn = sum(t * t for t in v)
        if vn == 0:
            continue
        for c in range(j, n):
            s = 2 * sum(v[i - j] * R[i][c] for i in range(j, m)) / vn
            for i in range(j, m):
                R[i][c] -= s * v[i - j]
        s = 2 * sum(v[i - j] * y[i] for i in range(j, m)) / vn
        for i in range(j, m):
            y[i] -= s * v[i - j]
    maxd = max(abs(R[j][j]) for j in range(n))
    if any(not abs(R[j][j]) > 1e-10 * maxd for j in range(n)):
        return None
    x = [0.0] * n
    for j in range(n - 1, -1, -1):
        x[j] = (y[j] - sum(R[j][c] * x[c] for c in range(j + 1, n))) / R[j][j]
    return x


def read_measured(path, points):
    """계측 CSV(열 = RPM, 차수, 응답점…) → [(rpm, 차수, 응답점, |계측|)]"""
    with open(path, encoding="utf-8-sig", newline="") as fp:
        rows = list(csv.reader(fp))
    head = [h.strip() for h in rows[0]]
    cols = {p: head.index(p) for p in points if p in head}
    out = []
    for r in rows[1:]:
        try:
            rpm, k = float(r[0]), float(r[1])
        except (ValueError, IndexError):
            continue
        for p, c in cols.items():
            try:
                out.append((rpm, k, p, abs(float(r[c]))))
            except (ValueError, IndexError):
                pass
    return out


def estimate(frfs, meas, interp, anti_ratio, degree=None, ratio=None):
    """계산 = |H_p(k·RPM/60)| × F_k(RPM) 와 계측의 제곱오차 합을 최소화.
    degree 가 있으면 차수별 다항식 계수, ratio 가 있으면 기준 차수의 RPM별 크기를 돌려줍니다."""
    obs = []
    for rpm, k, p, m in meas:
        freq, mag = frfs[p]
        h = interpolate(freq, mag, k * rpm / 60.0, interp)
        if h is None or not h > 0 or h < anti_ratio * max(mag):
            continue  # 범위 밖·반공진 부근은 맞춤에서 뺌 (웹 도구와 같음)
        obs.append((rpm, k, h, m))
    if ratio is not None:
        acc = {}
        for rpm, k, h, m in obs:
            a = h * ratio[k]
            n, d = acc.get(rpm, (0.0, 0.0))
            acc[rpm] = (n + a * m, d + a * a)
        return {rpm: n / d for rpm, (n, d) in sorted(acc.items())}
    xs = max(abs(o[0]) for o in obs) or 1.0
    fits = {}
    for k in sorted({o[1] for o in obs}):
        use = [o for o in obs if o[1] == k]
        A = [[h * (rpm / xs) ** j for j in range(degree + 1)] for rpm, _, h, _ in use]
        beta = lstsq(A, [m for *_, m in use])
        if beta is None:
            sys.exit("%g차: 계수가 정해지지 않습니다. 다항식 차수를 낮춰 주십시오." % k)
        fits[k] = [b / xs ** j for j, b in enumerate(beta)]
    return fits


## 2. 조건 입력 — 웹 도구에 넣은 값과 같게

In [ ]:
POINT = '예시_운전석바닥_진동'   # 응답점 이름
ORDERS = [1, 2, 4]               # 차수
FORCES = [120, 60, 25]           # 차수별 가진력
RPM_START, RPM_END, RPM_STEP = 800, 3000, 50
INTERP = 'linear'                # linear / nearest / loglog

# (선택) 차수별 scale factor — 쓰면 FORCES 대신 기준 차수 값 하나(REF_FORCE)만 넣습니다
#   F_k = (s_k / s_기준) × F_기준
SCALES = None                    # 예: [1, 0.5, 0.2]
REF_ORDER, REF_FORCE = 1, 120

## 3. 계산

In [ ]:
if SCALES:
    ratio = scale_ratios(ORDERS, SCALES, REF_ORDER)
    FORCES = [ratio[k] * REF_FORCE for k in ORDERS]
freq, mag = read_frf(FRF_CSV, POINT)
rows = []
for rpm in rpm_list(RPM_START, RPM_END, RPM_STEP):
    comps = []
    for k, F in zip(ORDERS, FORCES):
        h = interpolate(freq, mag, k * rpm / 60.0, INTERP)
        comps.append(None if h is None else h * F)
    s = [c for c in comps if c is not None]
    rows.append([rpm] + comps + [math.sqrt(sum(c * c for c in s)) if s else None])
for r in rows[:10]:
    print(r)

In [ ]:
with open('rpm_response_result.csv', 'w', encoding='utf-8-sig', newline='') as fp:
    w = csv.writer(fp)
    w.writerow(['RPM'] + ['%g차 응답' % k for k in ORDERS] + ['overall'])
    w.writerows(rows)
files.download('rpm_response_result.csv')

## 4. 가진력 추정 — 계산/계측 오차 최소화 (2026-09-29 추가)
계측 CSV(열 = RPM, 차수, 응답점 이름…)를 올리면 웹 도구 「4. 가진력 추정」과 같은 값을 냅니다.

- 계산 응답 = |FRF(차수 × RPM / 60)| × F_k(RPM), 목표 = Σ(계산 − 계측)² 최소 (크기 기준)
- **scale 고정**: F_k = (s_k/s_기준) × F_기준(RPM), RPM 점마다 F_기준 = Σ a·m ÷ Σ a² (a = |FRF| × s_k/s_기준)
- **형태 지정**: 차수마다 F_k(RPM) = c0 + c1·RPM + … + cn·RPMⁿ 를 QR 최소제곱으로

In [ ]:
up2 = files.upload()
MEAS_CSV = next(iter(up2))
EST_POINTS = ['예시_운전석바닥_진동', '예시_운전자귀_소음', '예시_핸들_진동']   # 계측과 짝지을 응답점
frfs = {p: read_frf(FRF_CSV, p) for p in EST_POINTS}
meas = read_measured(MEAS_CSV, EST_POINTS)

# (b) 형태 지정 — 차수별 다항식 계수
DEGREE = 1
for k, coef in estimate(frfs, meas, INTERP, 0.05, degree=DEGREE).items():
    print('%g차' % k, ['%.6g' % c for c in coef])

# (a) scale 고정 — 기준 차수의 RPM별 크기
ratio = scale_ratios([1, 2, 4], [1, 0.5, 0.208333], 1)
for rpm, F in estimate(frfs, meas, INTERP, 0.05, ratio=ratio).items():
    print(rpm, round(F, 4))

## 5. 계측 머리행 확인 — 차수 표기 정규화 (2026-09-29 저녁)

지점 이름은 파일에 있는 그대로 두고, 차수는 1차·2차 기본에 1·2, 1st·2nd, 1X, H1, order 1, Ord1, 1ord, ord1, 전각 숫자(１차)도 인식합니다.
알아보지 못한 머리행은 추측하지 않고 「인식 못함」으로 보여 줍니다. 웹 도구 「열 배정 확인」 표와 같은 규칙입니다(`python/rpm_response.py` 의 같은 함수).


In [ ]:
import re, csv, unicodedata

# ── 차수 표기 정규화 (2026-09-29 저녁 — 웹 도구 parseOrderLabel·splitMeasHeader 와 같은 규칙) ──
# 지점 이름은 파일 그대로, 차수는 1차·2차 기본 + 1·2, 1st·2nd, 1X, H1, order 1, Ord1, 1ord, ord1, 전각 숫자, 공백·대소문자 무시.
# 모르는 표기는 추측하지 않고 이유를 돌려줍니다(웹 화면은 「열 배정 확인」 표에 표시, 여기서는 표준오류로 알림).
GENERIC = "차수 표기를 알아보지 못했습니다"
# 1/rev 표기는 쓰지 않는다고 확인됨(2026-09-30) — 차수로 읽지 않음. 「/」가 구분자라 「1」+「rev」로 갈라지지 않게 막는다.
RE_PER_REV = re.compile(r"^\d+(?:\.\d+)?\s*(?:/|per)\s*rev(?:olutions?|s)?\.?$")
RE_PER_REV_IN = re.compile(r"\d\s*(?:/|per)\s*rev(?:olutions?|s)?(?![a-z])")
REV_REASON = "「n/rev」 표기는 쓰지 않는 것으로 확인했습니다(2026-09-30) — 1ord · ord1 · 1차 처럼 적거나 표에서 차수를 직접 골라 주십시오"
BARE_MAX = 200
SEP = r"\s_\-/|:·＿－／"
RE_EDGE = re.compile("^[" + SEP + "]+|[" + SEP + "]+$")
RE_SEPC = re.compile("[" + SEP + "]")


def _norm_label(t):
    s = unicodedata.normalize("NFKC", "" if t is None else str(t))
    s = re.sub(r"[\u00a0\u3000\s]+", " ", s).strip().lower()
    while True:
        u = re.sub(r"\s*(\([^()]*\)|\[[^\[\]]*\]|\{[^{}]*\})\s*$", "", s)
        if u == s or not u:
            break
        s = u.strip()
    w = re.match(r"^[(\[{]\s*(.*?)\s*[)\]}]$", s)
    return w.group(1) if w else s


def _ordinal_suffix(n):
    if 11 <= n % 100 <= 13:
        return "th"
    return {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")


def parse_order_label(t):
    """차수 표기 하나 → (차수 | 'overall' | None, 형태 또는 거부 이유)"""
    s = _norm_label(t)
    if not s:
        return None, "빈 칸"
    if re.match(r"^(overall|o\.?\s?a\.?|전체|합성)$", s):
        return "overall", "overall"

    def num(v, form):
        k = float(v)
        return (k, form) if k > 0 else (None, "차수는 0보다 커야 합니다")

    m = re.match(r"^(\d+(?:\.\d+)?)\s*차(?:수)?$", s) or re.match(r"^차수\s*[_\-#:]?\s*(\d+(?:\.\d+)?)$", s)
    if m:
        return num(m.group(1), "kr")
    m = re.match(r"^(\d+)\s*(st|nd|rd|th)(?:[\s_\-]*(?:order|ord\.?|차))?$", s)
    if m:
        if _ordinal_suffix(int(m.group(1))) != m.group(2):
            return None, "서수 접미사가 맞지 않습니다(%s%s 이어야 함)" % (m.group(1), _ordinal_suffix(int(m.group(1))))
        return num(m.group(1), "ordinal")
    if RE_PER_REV.match(s):
        return None, REV_REASON
    m = re.match(r"^(\d+(?:\.\d+)?)\s*x$", s)
    if m:
        return num(m.group(1), "x")
    m = re.match(r"^h\s*[_\-]?\s*(\d+)$", s)
    if m:
        return num(m.group(1), "h")
    m = re.match(r"^(?:order|ord)\.?\s*[_\-#:]?\s*(\d+(?:\.\d+)?)$", s) or re.match(r"^(\d+(?:\.\d+)?)\s*[_\-]?\s*(?:order|ord)\.?$", s)
    if m:
        return num(m.group(1), "order")
    if re.match(r"^\d+(?:\.\d+)?$", s):
        if float(s) > BARE_MAX:
            return None, "숫자만 있는 머리행은 %d 이하만 차수로 봅니다" % BARE_MAX
        return num(s, "num")
    return None, GENERIC


def split_meas_header(h, points=()):
    """머리행 → (지점 이름(파일 그대로, 없으면 ''), 차수 | 'overall' | None, 이유)"""
    t = ("" if h is None else str(h)).strip()
    if not t:
        return "", None, "빈 칸"
    if RE_PER_REV_IN.search(_norm_label(t)):
        return t, None, REV_REASON
    whole = RE_EDGE.sub("", t)
    k, why = parse_order_label(whole)
    if k is not None:
        return "", k, why
    for p in sorted([p for p in points if p], key=len, reverse=True):
        rest = None
        if t.startswith(p):
            rest = t[len(p):]
        elif len(t) > len(p) and t.endswith(p):
            rest = t[:len(t) - len(p)]
        if rest is None:
            continue
        k, _ = parse_order_label(RE_EDGE.sub("", rest))
        if k is not None:
            return p, k, ""
    cut = [i for i, ch in enumerate(t) if RE_SEPC.match(ch)]
    reason = None
    for c in cut:
        head, tail = RE_EDGE.sub("", t[:c]), RE_EDGE.sub("", t[c + 1:])
        if not head or not tail:
            continue
        k, r = parse_order_label(tail)
        if k is not None:
            return head, k, ""
        if r != GENERIC:
            reason = r
    for c in reversed(cut):
        head, tail = RE_EDGE.sub("", t[:c]), RE_EDGE.sub("", t[c + 1:])
        if not head or not tail:
            continue
        k, _ = parse_order_label(head)
        if k is not None:
            return tail, k, ""
    return t, None, reason or parse_order_label(whole)[1]


def _cell_order(v):
    """긴 형식의 차수 칸: 1 · 1차 · 1st · H1 · 1X … (overall·모르는 표기는 None)"""
    k, _ = parse_order_label(v)
    return None if k == "overall" else k


# 4번에서 올린 계측 CSV 의 머리행을 열마다 읽어 봅니다
with open(MEAS_CSV, encoding='utf-8-sig', newline='') as fp:
    head = next(csv.reader(fp))
for c, h in enumerate(head[1:], start=2):
    pt, k, why = split_meas_header(h, EST_POINTS)
    print('%2d열  %-32s → %s' % (c, h, ('지점 「%s」 · ' % pt if pt else '') + ('overall' if k == 'overall' else '%g차' % k) if k is not None else '인식 못함 — ' + why))